# 2. Connectivities

On an unstructured mesh, the neighbors of a point are not found by offsets but through **relations**, e.g. from each cell to its edges. This notebook covers the connectivity concepts of `concepts.py` and their use with fields; DSL operators using them come in the next notebooks. Chains of reads and partial relations are in "8. Chains and partial relations", the footprint algebra and pairs in "9. Footprint algebra and pairs".

Contents: relations and neighbor dimensions · neighbor tables and meshes · reading through a relation · `sum_over` and local fields · domains through relations.

The example mesh: a quadrilateral `c0` and two triangles `c1` and `c2`, with 8 edges and 6 vertices.

```
v3 ---e5--- v4 ---e6--- v5
|           |  c2     / |
e2   c0     e3     e7   e4
|           |   /   c1  |
v0 ---e0--- v1 ---e1--- v2
```

## Relations and neighbor dimensions

| Concept | Kind | Meaning |
|---|---|---|
| `Relation[S, T]` | class | A relation from each point of the source dimension `S` to its neighbors along `T`, both `Location` dimensions (an `Axis`, `Label` or `Neighbor` is a `TypeError`). Declared by subclassing it, with `max_neighbors` and `skip_values` (whether some rows have fewer neighbors); `partial` relations, with neighbors missing at the boundary of the mesh, are in "8. Chains and partial relations". A type that only *names* the connectivity: never instantiated |
| `is_relation` | function | Whether an object is a relation |
| `Neighbor[R]` | dimension | The neighbor dimension of `R`, the fourth kind: a point of `Neighbor[R]` is a position in a row of the table of `R`. One class per relation and parent, the dimension it hangs from, created on first use: `Neighbor[C2E]` is `Neighbor[C2E, Cell]` (chains of reads: see 8) |
| `neighbor_of` | function | `neighbor_of(R)` is `Neighbor[R]`, for a relation `R` only known at runtime |

A point of `Neighbor[R]` is a *position in a row*, not a point of `R.target`: an inner edge is in the rows of two cells, usually at different positions, and a skip position is not a neighbor of anything. Reading a field through `C2E` adds the dimension `Neighbor[C2E]`; summing over it removes it.

In [1]:
import numpy as np
from concepts import Axis, DimensionIndex, Location, Neighbor, Relation, is_relation, neighbor_of


class Cell(Location): ...


class Edge(Location): ...


class Vertex(Location): ...


class K(Axis): ...  # vertical levels


class C2E(Relation[Cell, Edge], max_neighbors=4, skip_values=True): ...  # triangles: 3 edges


class E2C(Relation[Edge, Cell], max_neighbors=2, skip_values=True): ...  # boundary: 1 cell


class E2V(Relation[Edge, Vertex], max_neighbors=2): ...  # always 2 vertices


for r in (C2E, E2C, E2V):
    print(
        f"{r.__name__}: {r.source.__name__} -> {r.target.__name__}, max {r.max_neighbors}, skip values {r.skip_values}"
    )
print("is_relation:", is_relation(C2E), is_relation(Cell))
print(
    "Neighbor[C2E]:",
    Neighbor[C2E].__name__,
    "| interned:",
    Neighbor[C2E] is Neighbor[C2E] is neighbor_of(C2E),
)
print("a dimension:", issubclass(Neighbor[C2E], DimensionIndex), "| position:", Neighbor[C2E].at(2))
print(
    "parent:",
    Neighbor[C2E].parent.__name__,
    "| Neighbor[C2E, Cell]:",
    Neighbor[C2E] is Neighbor[C2E, Cell],
)
try:
    C2E()
except TypeError as error:
    print("TypeError:", error)
try:

    class Bad(Relation[Cell, K], max_neighbors=1): ...

except TypeError as error:
    print("TypeError:", error)

C2E: Cell -> Edge, max 4, skip values True
E2C: Edge -> Cell, max 2, skip values True
E2V: Edge -> Vertex, max 2, skip values False
is_relation: True False
Neighbor[C2E]: Neighbor[C2E] | interned: True
a dimension: True | position: Neighbor[C2E](2)
parent: Cell | Neighbor[C2E, Cell]: True
TypeError: A relation is a type: it is never instantiated.
TypeError: 'K' cannot be in a relation: use a `Location`.


## Neighbor tables and meshes

| Concept | Kind | Meaning |
|---|---|---|
| `NeighborTable` | alias | An integer `NDArrayField` with dimensions `(R.source, Neighbor[R])`: at `(s, k)`, the index of the `k`-th neighbor of `s` |
| `SKIP_VALUE` | constant | `-1`: a *skip position*, no neighbor at that position of the row (only if `R.skip_values`) |
| `check_neighbor_table` | function | Raises a `ValueError` if a table does not fit a relation (dimensions, number of neighbors, indices), and, with `extents`, its rows and indices |
| `Mesh` | class | A read-only mapping from relations to their tables, each checked once, when the mesh is built, with optional `extents`, the number of points of some dimensions, e.g. `{Cell: 3}`: the row counts and indices of the tables are checked against them. A context manager: `with Mesh({C2E: table, ...}, extents={...}):`, or `with m:` for a mesh `m`, makes the tables available to the code in the block, e.g. DSL operators |
| `current_mesh()` | function | The `Mesh` of the innermost `with` block of a mesh: an empty one outside |

The tables are only known at runtime, while the relations are types: the same operator runs on any mesh. An inner `with` block, of another mesh or of the same one, replaces the outer mesh until it ends; each thread has its own current mesh. `neighbor_table(R, rows)` from `implementation.py` builds a table from a list of rows. A second sentinel, `MISSING_VALUE`, marks neighbors missing at the boundary of the mesh: see "8. Chains and partial relations".

In [2]:
from concepts import SKIP_VALUE, Mesh, check_neighbor_table, current_mesh
from implementation import neighbor_table

X = SKIP_VALUE
c2e = neighbor_table(C2E, [[0, 3, 5, 2], [1, 4, 7, X], [7, 6, 3, X]])
e2c = neighbor_table(E2C, [[0, X], [1, X], [0, X], [0, 2], [1, X], [0, X], [2, X], [1, 2]])
e2v = neighbor_table(E2V, [[0, 1], [1, 2], [0, 3], [1, 4], [2, 5], [3, 4], [4, 5], [1, 5]])
tables = {C2E: c2e, E2C: e2c, E2V: e2v}
print("c2e =", c2e)

for relation, bad, extents in [
    (E2V, neighbor_table(E2V, [[0, X]]), None),
    (C2E, e2v, None),
    (E2V, neighbor_table(E2V, [[0, 6]]), {Vertex: 6}),  # there is no vertex 6
]:
    try:
        check_neighbor_table(relation, bad, extents)
    except ValueError as error:
        print("ValueError:", error)

with Mesh(tables, extents={Cell: 3, Edge: 8, Vertex: 6}) as triangles_and_quad:
    print(
        "Mesh:",
        isinstance(triangles_and_quad, Mesh),
        "| relations:",
        [r.__name__ for r in triangles_and_quad],
    )
    with Mesh({E2V: e2v}):
        print("inner block:", [r.__name__ for r in current_mesh()])
        try:
            current_mesh()[C2E]
        except LookupError as error:
            print("LookupError:", error)
    print("outer again:", len(current_mesh()))
    with triangles_and_quad as same:  # entered again, inside itself: not checked again
        print("same mesh:", same is triangles_and_quad)
print("outside:", len(current_mesh()))

c2e = Field(Cell[0:3], Neighbor[C2E][0:4]: [[0, 3, 5, 2], [1, 4, 7, -1], [7, 6, 3, -1]])
ValueError: The table of 'E2V' has invalid neighbor indices.
ValueError: The table of 'C2E' must have the dimensions (<class '__main__.Cell'>, <class '__main__.Neighbor[C2E]'>).
ValueError: The table of 'E2V' has neighbor indices beyond the extent of 'Vertex', 6.
Mesh: True | relations: ['C2E', 'E2C', 'E2V']
inner block: ['E2V']
LookupError: No neighbor table for 'C2E': use `with Mesh(...)` to provide it.
outer again: 3
same mesh: True
outside: 0


## Reading through a relation

`e(C2E)`, with `e: Field[Dims[Edge], float]`, premaps `e` by the relation: a **gather** through the table of `C2E` in the current mesh. The result has the dimensions `(Cell, Neighbor[C2E])`, followed by the other dimensions of `e`, e.g. `K`, in order: its value at `(c, n)` is the value of `e` at the `n`-th edge of `c`.

- Its domain along `Cell` is computed forward: the cells whose edges are all in the domain of `e`, as a range or an index set.
- Its values at skip positions are unspecified (0 in `implementation.py`): reductions like `sum_over` ignore them.
- Offsets and relations can be combined in a premap, applied in order: `e(C2E, K + 1)`; `v(E2V, C2E)` is `v(E2V)(C2E)`.

In [3]:
from implementation import NumPyArrayField

length = NumPyArrayField({Edge: range(8)}, [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.41])
with Mesh(tables):
    print("length(C2E) =", length(C2E))
    left_edges = length[{Edge: (0, 2, 3, 5)}]  # only the edges of c0
    print("partial     =", left_edges(C2E))
    print("vertices    =", NumPyArrayField({Vertex: range(6)}, np.arange(6.0))(E2V))

    level_flux = NumPyArrayField({Edge: range(8), K: range(3)}, np.arange(24.0).reshape(8, 3))
    gathered = level_flux(C2E, K + 1)
    print("with K + 1  :", [d.__name__ for d in gathered.domain], "| K", gathered.domain[K])

length(C2E) = Field(Cell[0:3], Neighbor[C2E][0:4]: [[1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.41, 0.0], [1.41, 1.0, 1.0, 0.0]])
partial     = Field(Cell[0:1], Neighbor[C2E][0:4]: [[1.0, 1.0, 1.0, 1.0]])
vertices    = Field(Edge[0:8], Neighbor[E2V][0:2]: [[0.0, 1.0], [1.0, 2.0], [0.0, 3.0], [1.0, 4.0], [2.0, 5.0], [3.0, 4.0], [4.0, 5.0], [1.0, 5.0]])
with K + 1  : ['Cell', 'Neighbor[C2E]', 'K'] | K range(-1, 2)


## `sum_over` and local fields

`sum_over(f, axis=Neighbor[R])`, a DSL builtin of `implementation.py`, sums `f` over the neighbor dimension of `R`, skipping the skip positions (0 for a point without neighbors): the result has no `Neighbor[R]` dimension. Likewise, `max_over(f, axis=Neighbor[R])` and `min_over(f, axis=Neighbor[R])` give the largest and the smallest value, skipping the skip positions: for a point without neighbors, `-inf` and `+inf` (the lowest and the highest integer for ints). Outside programs, `axis` must be a `Neighbor` dimension (else a `TypeError`): sums over the points of a distributed field are `global_sum` in local programs (see "10. Local programs"), or `sum_over` over a `Location` in a global program (see "13. Global programs").

A **local field** has a neighbor dimension itself, e.g. `Field[Dims[Cell, Neighbor[C2E]], float]`: one value per cell and edge *of that cell*, like the orientation of an edge relative to the cell, `+1` or `-1`. Local fields combine with gathered fields point by point, and are then reduced.

In [4]:
from implementation import max_over, min_over, sum_over

edge_ones = NumPyArrayField({Edge: range(8)}, np.ones(8))
# the flux through each edge, along the normal of the edge, and the orientation of each edge
# of a cell: +1 if the normal of the edge points out of the cell, else -1 (0 at skip positions)
flux = NumPyArrayField({Edge: range(8)}, [2.0, 1.0, 0.0, 1.0, 0.5, 2.0, 1.0, 0.5])
orientation = NumPyArrayField(
    {Cell: range(3), Neighbor[C2E]: range(4)},
    [[-1.0, 1.0, 1.0, -1.0], [-1.0, 1.0, -1.0, 0.0], [1.0, 1.0, -1.0, 0.0]],
)
with Mesh(tables):
    perimeter = sum_over(length(C2E), axis=Neighbor[C2E])
    edges_per_cell = sum_over(edge_ones(C2E), axis=Neighbor[C2E])
    print("perimeter      =", perimeter)
    print("edges per cell =", edges_per_cell)
    print("net outflow    =", sum_over(orientation * flux(C2E), axis=Neighbor[C2E]))
    # cell 0: edges 0, 3, 5, 2, all 1.0; cells 1 and 2: edge 7, 1.41 (the skip position skipped)
    print("longest edge   =", max_over(length(C2E), axis=Neighbor[C2E]))
    # cell 0: edges 0, 3, 5, 2, flux 2, 1, 2, 0 -> 0; cell 1: 1, 4, 7 -> 1, 0.5, 0.5 -> 0.5;
    # cell 2: 7, 6, 3 -> 0.5, 1, 1 -> 0.5
    print("lowest flux    =", min_over(flux(C2E), axis=Neighbor[C2E]))
    cell_area = NumPyArrayField({Cell: range(3)}, [1.0, 0.5, 0.5])
    print("area by edge   =", sum_over(cell_area(E2C), axis=Neighbor[E2C]))  # 1 or 2 cells

perimeter      = Field(Cell[0:3]: [4.0, 3.41, 3.41])
edges per cell = Field(Cell[0:3]: [4.0, 3.0, 3.0])
net outflow    = Field(Cell[0:3]: [1.0, -1.0, 0.5])
longest edge   = Field(Cell[0:3]: [1.0, 1.41, 1.41])
lowest flux    = Field(Cell[0:3]: [0.0, 0.5, 0.5])
area by edge   = Field(Edge[0:8]: [1.0, 0.5, 1.0, 1.5, 0.5, 1.0, 0.5, 1.0])


## Domains through relations

The `Extents` of a DSL operator are chains of reads: `sum_over(e(C2E), axis=Neighbor[C2E])` reads `e` with `(C2E, Reduction(Neighbor[C2E]))`, through `C2E` along `Edge`, then summed. Domains then propagate through the tables of the current mesh, with two helpers of `implementation.py`:

- **forward**, `sources_within(R, targets)`: the sources whose neighbors are all in `targets`, e.g. the cells that can be computed from some edges;
- **backward**, `neighbors(R, sources)`: all the neighbors of `sources`, e.g. the edges needed to compute some cells.

Both usually give index sets. The backends use them when running DSL operators, e.g. `lazy_backward` computes only the edges needed for a requested set of cells: see "3. DSL definitions and operators" for the extents and "5. Processors and backends" for the backends.

In [5]:
from implementation import neighbors, sources_within

with Mesh(tables):
    print("edges of c1, c2:", neighbors(C2E, (1, 2)))  # needed to compute c1 and c2
    print("cells within e0-e3, e5:", sources_within(C2E, (0, 1, 2, 3, 5)))  # only c0
    print(
        "vertices of e7:",
        neighbors(E2V, (7,)),
        "| edges within v0, v1, v3:",
        sources_within(E2V, (0, 1, 3)),
    )

edges of c1, c2: (1, 3, 4, 6, 7)
cells within e0-e3, e5: range(0, 1)
vertices of e7: (1, 5) | edges within v0, v1, v3: (0, 2)
